
# MLOps on Gemini Enterprise Agent Platform
### GCP Training Program — Module 7 (3.5 Hours): Pipelines, Model Registry, CI/CD, Model Monitoring

**Problem statement:** we're picking up exactly where the Day 4 churn-prediction notebook left off.
There, we trained and registered a churn model **by hand** — one notebook cell at a time, one
Custom Training job, one HPT job, manually calling `Model.upload()`. That's fine for exploration,
but it isn't **repeatable, versioned, or automatable**. This notebook turns that same churn problem
into a proper **MLOps workflow**:

| Section | What we build | Answers the question |
|---|---|---|
| 3 | **Vertex AI Pipelines** | A single, reusable, versioned definition of "load data → train → conditionally register → deploy" |
| 4 | **Model Registry** | Where every model version this pipeline produces actually lives, and how to inspect them |
| 5 | **CI/CD for ML** | How a code change triggers a full pipeline run, deploys as canary traffic, gets promoted, and — critically — can be **rolled back** |
| 6 | **Model Monitoring** | How we'd know if a model quietly started drifting in production, without waiting for a complaint |

We reuse the **same 100-row customer churn dataset** from Day 4
(`https://github.com/himanshurathi/gcp-training-labs-accordion/blob/master/day-4-vertex/customer_dataset.csv`)
specifically so the class isn't learning a new problem *and* new plumbing at the same time — the
dataset is already familiar; everything new here is process.

## ⚠️ Terminology note
Everything below lives under **Agent Platform > Models** in the console (Pipelines, Model Registry,
Deploy, Monitoring), matching the Day 4 notebook's rebrand note — code still uses `aiplatform`/`kfp`
package names unchanged.

## ⚠️ Cost & time notes
- **Pipeline runs** (Sections 3, 5) typically take **10-20 minutes each** — Vertex AI Pipelines has
  the same per-step VM-provisioning overhead we saw with Custom Training jobs in Day 4, multiplied
  across several pipeline steps run partly in sequence.
- **The deployed Endpoint** (Sections 3, 5, 6) bills continuously once created — **Section 7's
  cleanup is not optional**, it's the single most important cell in this notebook to actually run.
- This notebook is fully self-contained: authenticate → configure → everything else runs against
  your own project.


## 1. Authenticate & Install

In [1]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")


Authenticated in Colab.


In [2]:

!pip install --upgrade --quiet "google-cloud-aiplatform>=1.70" "kfp>=2.7" "google-cloud-pipeline-components>=2.15" "google-cloud-build" pandas scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.9/69.9 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.9/477.9 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.3/187.3 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 78.0 MB/s eta 0:00:00
   ━━━━━

In [3]:

import IPython
IPython.Application.instance().kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

## 2. Configure Your Project & Region

In [1]:

import time

PROJECT_ID = input("Enter your GCP Project ID: ").strip()
REGION = input("Enter your Agent Platform region [default: us-central1]: ").strip() or "us-central1"

_suffix = int(time.time())
BUCKET_NAME = f"{PROJECT_ID}-mlops-lab-{_suffix}"
BUCKET_URI = f"gs://{BUCKET_NAME}"
PIPELINE_ROOT = f"{BUCKET_URI}/pipeline-root"

!gcloud config set project {PROJECT_ID}
!gcloud services enable aiplatform.googleapis.com storage.googleapis.com cloudbuild.googleapis.com --project {PROJECT_ID}

from google.cloud import storage
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.lookup_bucket(BUCKET_NAME)
if bucket is None:
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Created bucket {BUCKET_URI}")

from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

GITHUB_CSV_URL = (
    "https://raw.githubusercontent.com/himanshurathi/"
    "gcp-training-labs-accordion/master/day-4-vertex/customer_dataset.csv"
)

print("Project:", PROJECT_ID)
print("Region:", REGION)
print("Pipeline root:", PIPELINE_ROOT)


Enter your GCP Project ID: cs-poc-ytkjb6nscjlihtluxkcmuoq
Enter your Agent Platform region [default: us-central1]: 
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey

Operation "operations/acat.p2-226858601047-4c89cd3f-6e02-433f-84a5-54833bfdd751" finished successfully.
Created bucket gs://cs-poc-ytkjb6nscjlihtluxkcmuoq-mlops-lab-1783615448
Project: cs-poc-ytkjb6nscjlihtluxkcmuoq
Region: us-central1
Pipeline root: gs://cs-poc-ytkjb6nscjlihtluxkcmuoq-mlops-lab-1783615448/pipeline-root



## 3. Vertex AI Pipelines

### 3.1 What a Pipeline Actually Is
A pipeline is a **directed acyclic graph (DAG)** of *components* — small, independent, containerized
steps, each with declared inputs and outputs. Vertex AI Pipelines (built on the open-source Kubeflow
Pipelines, or **KFP**, SDK) takes that DAG definition, compiles it to JSON, and runs each step as a
managed job, automatically passing outputs from one step as inputs to the next. This is the exact
same underlying mechanism as the Custom Training job from Day 4 — except now it's several such steps
wired together and repeatable with one API call, instead of one job you configured by hand.

We'll define **4 components**:

| # | Component | Input | Output |
|---|---|---|---|
| 1 | `load_data` | GitHub CSV URL | a `Dataset` artifact |
| 2 | `train_model` | the `Dataset` artifact, hyperparameters | a `Model` artifact (the trained pipeline) + an accuracy `Metrics` artifact |
| 3 | `register_model` | the `Model` artifact | the Vertex AI Model Registry resource name (a string) |
| 4 | `deploy_model` | the registered model's resource name, an endpoint name, a traffic-split percentage | nothing (a side effect: the model becomes servable) |

Step 3 only runs if step 2's accuracy clears a threshold — this is the pipeline's built-in
**conditional logic**, so a bad training run can't silently get deployed.



### 3.2 Component 1 — `load_data`
**What this does:** the simplest possible pipeline step — read the CSV, output it as a KFP
`Dataset` artifact so downstream components can consume it without knowing anything about where it
came from. Using `@dsl.component` (rather than `google_cloud_pipeline_components`' prebuilt ops)
means this step runs in a lightweight, disposable container — no custom Docker image to build or
push, ideal for a small, class-friendly custom step like this one.


In [2]:

from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics

@dsl.component(base_image="python:3.11", packages_to_install=["pandas==2.2.2"])
def load_data(csv_url: str, output_dataset: Output[Dataset]):
    import pandas as pd
    df = pd.read_csv(csv_url)
    df.to_csv(output_dataset.path, index=False)
    print(f"Loaded {len(df)} rows from {csv_url}")



### 3.3 Component 2 — `train_model`
**What this does:** the same Random Forest training logic from Day 4's `train.py`, now as a pipeline
component. It writes the trained model to a KFP `Model` artifact (which Vertex AI Pipelines
automatically places in Cloud Storage under `PIPELINE_ROOT`) and records the accuracy as a
`Metrics` artifact — this is what step 4's conditional logic will check.


In [7]:
from typing import NamedTuple

@dsl.component(
    base_image="python:3.11",
    packages_to_install=["pandas==2.2.2", "scikit-learn", "joblib"],
)
def train_model(
    input_dataset: Input[Dataset],
    n_estimators: int,
    max_depth: int,
    output_model: Output[Model],
    output_metrics: Output[Metrics],
) -> NamedTuple("Outputs", [("accuracy", float)]):
    import pandas as pd
    import joblib
    import os
    from collections import namedtuple
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.metrics import accuracy_score, f1_score

    df = pd.read_csv(input_dataset.path)
    feature_cols = ['age', 'income', 'region', 'membership_tier', 'tenure_months',
                    'num_purchases', 'satisfaction_score']
    X = df[feature_cols]
    y = df['churned']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    preprocess = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['region', 'membership_tier']),
    ], remainder='passthrough')

    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    pipe.fit(X_train, y_train)

    preds = pipe.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    # Metrics artifact: for the Pipelines UI's visualization tab only.
    output_metrics.log_metric("accuracy", accuracy)
    output_metrics.log_metric("f1_score", f1)
    print(f"accuracy={accuracy:.4f} f1={f1:.4f}")

    os.makedirs(output_model.path, exist_ok=True)
    joblib.dump(pipe, os.path.join(output_model.path, "model.joblib"))

    # Scalar output parameter: this is what the pipeline's dsl.Condition can actually compare.
    outputs = namedtuple("Outputs", ["accuracy"])
    return outputs(accuracy=accuracy)


### 3.4 Component 3 — `register_model`
**What this does:** uploads the trained model artifact to the **Model Registry** under a fixed
`model_display_name`. Registering multiple times under the **same display name** is exactly what
creates **model versions** — Section 4 shows this directly. We pass the prebuilt scikit-learn
serving container from Day 4, since we're serving the same kind of model artifact.


In [8]:

@dsl.component(
    base_image="python:3.11",
    packages_to_install=["google-cloud-aiplatform"],
)
def register_model(
    project: str,
    region: str,
    model_display_name: str,
    input_model: Input[Model],
) -> str:
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)

    uploaded_model = aiplatform.Model.upload(
        display_name=model_display_name,
        artifact_uri=input_model.uri,
        serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest",
    )
    print(f"Registered model version: {uploaded_model.resource_name}")
    print(f"Version ID: {uploaded_model.version_id}")
    return uploaded_model.resource_name



### 3.5 Component 4 — `deploy_model`
**What this does:** deploys the registered model to an **Endpoint**, either creating that endpoint
fresh (first run) or deploying alongside whatever's already there with a specific **traffic split**
percentage (subsequent runs — this is what Section 5's canary/rollback demo relies on).


In [5]:

@dsl.component(
    base_image="python:3.11",
    packages_to_install=["google-cloud-aiplatform"],
)
def deploy_model(
    project: str,
    region: str,
    model_resource_name: str,
    endpoint_display_name: str,
    deployed_model_display_name: str,
    traffic_percentage: int,
    machine_type: str,
) -> str:
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)

    model = aiplatform.Model(model_resource_name)

    existing_endpoints = aiplatform.Endpoint.list(
        filter=f'display_name="{endpoint_display_name}"'
    )
    if existing_endpoints:
        endpoint = existing_endpoints[0]
        print(f"Reusing existing endpoint: {endpoint.resource_name}")
    else:
        endpoint = aiplatform.Endpoint.create(display_name=endpoint_display_name)
        print(f"Created new endpoint: {endpoint.resource_name}")

    # traffic_split maps deployed-model-id -> percentage; existing deployed models not listed
    # here get their remaining traffic auto-adjusted by the API to make the split add up to 100.
    model.deploy(
        endpoint=endpoint,
        deployed_model_display_name=deployed_model_display_name,
        machine_type=machine_type,
        min_replica_count=1,
        max_replica_count=1,
        traffic_percentage=traffic_percentage,
    )
    print(f"Deployed with {traffic_percentage}% traffic to endpoint {endpoint.resource_name}")
    return endpoint.resource_name



### 3.6 Assemble the Pipeline
**What this does:** wires the four components into one `@dsl.pipeline` function. The
`dsl.Condition` block is the conditional-deployment logic — `register_model` and `deploy_model`
only execute if `train_model`'s logged accuracy clears `accuracy_threshold`.


In [9]:

@dsl.pipeline(
    name="churn-model-pipeline",
    description="Load data, train, conditionally register and deploy the churn model.",
)
def churn_pipeline(
    csv_url: str,
    project: str,
    region: str,
    n_estimators: int = 200,
    max_depth: int = 5,
    accuracy_threshold: float = 0.60,
    model_display_name: str = "churn-pipeline-model",
    endpoint_display_name: str = "churn-pipeline-endpoint",
    deployed_model_display_name: str = "churn-v1",
    traffic_percentage: int = 100,
    machine_type: str = "e2-standard-4",
):
    load_task = load_data(csv_url=csv_url)

    train_task = train_model(
        input_dataset=load_task.outputs["output_dataset"],
        n_estimators=n_estimators,
        max_depth=max_depth,
    )

    with dsl.Condition(
        train_task.outputs["accuracy"] >= accuracy_threshold,
        name="accuracy-check",
    ):
        register_task = register_model(
            project=project,
            region=region,
            model_display_name=model_display_name,
            input_model=train_task.outputs["output_model"],
        )

        deploy_model(
            project=project,
            region=region,
            model_resource_name=register_task.output,
            endpoint_display_name=endpoint_display_name,
            deployed_model_display_name=deployed_model_display_name,
            traffic_percentage=traffic_percentage,
            machine_type=machine_type,
        )

print("Pipeline defined.")


Pipeline defined.


/tmp/ipykernel_845/1134743722.py:26: DeprecationWarning: dsl.Condition is deprecated. Please use dsl.If instead.
  with dsl.Condition(



### 3.7 Compile the Pipeline
**What this does:** turns the Python function above into a portable JSON pipeline definition — the
artifact that actually gets submitted to Vertex AI Pipelines, and the same file Section 5's CI/CD
step will compile and submit automatically instead of by hand.


In [10]:

from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=churn_pipeline,
    package_path="churn_pipeline.json",
)
print("Compiled to churn_pipeline.json")


Compiled to churn_pipeline.json



### 3.8 Submit the First Pipeline Run (v1 — Full Deploy)
**What this does:** submits the compiled pipeline as a `PipelineJob`, with `traffic_percentage=100`
since this is the very first deployment — there's no existing model to share traffic with yet.
This run will take **10-20 minutes** — most of it is step-by-step VM provisioning across all four
components running mostly in sequence.


In [11]:

pipeline_job_v1 = aiplatform.PipelineJob(
    display_name="churn-pipeline-v1",
    template_path="churn_pipeline.json",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "csv_url": GITHUB_CSV_URL,
        "project": PROJECT_ID,
        "region": REGION,
        "n_estimators": 200,
        "max_depth": 5,
        "accuracy_threshold": 0.60,
        "model_display_name": "churn-pipeline-model",
        "endpoint_display_name": "churn-pipeline-endpoint",
        "deployed_model_display_name": "churn-v1",
        "traffic_percentage": 100,
        "machine_type": "e2-standard-4",
    },
)

pipeline_job_v1.submit()
print("Pipeline v1 run state:", pipeline_job_v1.state)


KeyboardInterrupt: 


> 🖥️ **Check it in the console:** **Agent Platform > Models > Pipelines** → open
> `churn-pipeline-v1` → the DAG view shows all four steps (and the conditional branch) as a visual
> graph, colored by status — click any step to see its logs directly, without hunting through
> Cloud Logging.



## 4. Model Registry

**What this does:** confirms the pipeline's `register_model` step actually created a Model Registry
entry, and shows what "version" means concretely — every model registered under the same
`model_display_name` becomes a new **version** of one logical model, not a separate model.


In [12]:

registered_model = aiplatform.Model.list(filter='display_name="churn-pipeline-model"')[0]
print("Model resource name:", registered_model.resource_name)
print("Current version ID:", registered_model.version_id)

versions = registered_model.versioning_registry.list_versions()
print(f"\nAll versions under this display name ({len(versions)} so far):")
for v in versions:
    print(f"  - version {v.version_id}, created {v.version_create_time}")


Model resource name: projects/226858601047/locations/us-central1/models/6967790452925792256
Current version ID: 1

All versions under this display name (1 so far):
  - version 1, created 2026-07-09 16:57:09.929691+00:00



> 🖥️ **Check it in the console:** **Agent Platform > Models > Model Registry** →
> `churn-pipeline-model` → the **Versions** tab lists every version, and the **Deploy & Test** tab
> shows which endpoint(s) each version is currently deployed to with what traffic split.



## 5. CI/CD for ML

### 5.1 The CI Trigger — What Cloud Build Would Run
**What this does:** in a real project, a `git push` to your ML repo would trigger a Cloud Build
job that compiles and submits this pipeline automatically — no one manually re-running notebook
cells. The script below is exactly that CI script, written to disk as a concrete, illustrative
artifact for the class to inspect. **We don't execute it remotely in this notebook** (that would
need a real source repo and Cloud Build trigger wired up ahead of time); instead, Section 5.2 runs
the identical compile-and-submit logic directly, so the mechanics are demonstrated live without
needing that additional CI infrastructure set up first.


In [13]:

import os

os.makedirs("ci_pipeline", exist_ok=True)

with open("ci_pipeline/submit_pipeline.py", "w") as f:
    f.write('''
import sys
from google.cloud import aiplatform
from kfp import compiler
sys.path.insert(0, ".")
from pipeline_def import churn_pipeline

PROJECT_ID = sys.argv[1]
REGION = sys.argv[2]
PIPELINE_ROOT = sys.argv[3]
CSV_URL = sys.argv[4]
DEPLOYED_MODEL_DISPLAY_NAME = sys.argv[5]
TRAFFIC_PERCENTAGE = int(sys.argv[6])
N_ESTIMATORS = int(sys.argv[7])

aiplatform.init(project=PROJECT_ID, location=REGION)

compiler.Compiler().compile(pipeline_func=churn_pipeline, package_path="churn_pipeline.json")

job = aiplatform.PipelineJob(
    display_name=f"churn-pipeline-ci-{DEPLOYED_MODEL_DISPLAY_NAME}",
    template_path="churn_pipeline.json",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "csv_url": CSV_URL,
        "project": PROJECT_ID,
        "region": REGION,
        "n_estimators": N_ESTIMATORS,
        "max_depth": 5,
        "accuracy_threshold": 0.60,
        "model_display_name": "churn-pipeline-model",
        "endpoint_display_name": "churn-pipeline-endpoint",
        "deployed_model_display_name": DEPLOYED_MODEL_DISPLAY_NAME,
        "traffic_percentage": TRAFFIC_PERCENTAGE,
        "machine_type": "e2-standard-4",
    },
)
job.run(sync=True)
print("CI-triggered pipeline run complete:", job.state)
''')

print("Wrote ci_pipeline/submit_pipeline.py — this is what a Cloud Build trigger would run on a")
print("git push in a real setup. Section 5.2 below runs the equivalent logic live, directly.")


Wrote ci_pipeline/submit_pipeline.py — this is what a Cloud Build trigger would run on a
git push in a real setup. Section 5.2 below runs the equivalent logic live, directly.



### 5.2 Release v2 Through the Same Pipeline — as a Canary
**What this does:** submits the **same pipeline definition**, changing only `n_estimators` (standing
in for "a code/config change was pushed") and `deployed_model_display_name="churn-v2"`, with
`traffic_percentage=20` — this deploys v2 to receive **20% of traffic**, automatically leaving the
remaining 80% on the already-deployed v1. This is a **canary release**: a small, safe fraction of
real traffic on the new version before fully committing to it.


In [15]:

# In a real CI/CD setup, Cloud Build would run the script written above (ci_pipeline/submit_pipeline.py)
# in response to a git push, using its own copy of the pipeline definition. Here, we invoke the
# identical compile-and-submit steps directly against the already-defined `churn_pipeline` function
# from Section 3.6 — same mechanics, just triggered by this cell instead of a Cloud Build trigger.
compiler.Compiler().compile(pipeline_func=churn_pipeline, package_path="churn_pipeline.json")

pipeline_job_v2 = aiplatform.PipelineJob(
    display_name="churn-pipeline-v2-canary",
    template_path="churn_pipeline.json",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "csv_url": GITHUB_CSV_URL,
        "project": PROJECT_ID,
        "region": REGION,
        "n_estimators": 350,  # the "code change" being released
        "max_depth": 6,
        "accuracy_threshold": 0.60,
        "model_display_name": "churn-pipeline-model",
        "endpoint_display_name": "churn-pipeline-endpoint",
        "deployed_model_display_name": "churn-v2",
        "traffic_percentage": 20,
        "machine_type": "e2-standard-4",
    },
)

pipeline_job_v2.submit()
print("Pipeline v2 (canary) run state:", pipeline_job_v2.state)


Pipeline v2 (canary) run state: 2



> 🖥️ **Check it in the console:** **Agent Platform > Models > Model Registry > churn-pipeline-model
> > Deploy & Test** → you should now see **two versions** deployed to the same endpoint —
> `churn-v1` at ~80% traffic, `churn-v2` at ~20%.

### 5.3 Verify Both Versions Are Actually Serving


In [16]:

endpoint = aiplatform.Endpoint(
    aiplatform.Endpoint.list(filter='display_name="churn-pipeline-endpoint"')[0].resource_name
)
print("Current traffic split (deployed_model_id -> percentage):")
print(endpoint.traffic_split)

sample_instance = [35, 60000.0, "East", "Silver", 24, 12, 7.5]
for i in range(6):
    prediction = endpoint.predict(instances=[sample_instance])
    print(f"Call {i+1}: deployed_model_id served = {prediction.deployed_model_id}")


Current traffic split (deployed_model_id -> percentage):
{}


FailedPrecondition: 400 Endpoint projects/226858601047/locations/us-central1/endpoints/3303311711594872832 misconfigured, "traffic_split" not set.  Verify if any models are deployed to the endpoint and traffic split is configured for them.


### 5.4 Promote v2 to 100% Traffic
**What this does:** once we're confident in v2's canary performance, shift all traffic to it —
this is the "graduate the release" step, done purely via a traffic-split update, no retraining
needed.


In [17]:

v2_deployed_id = [
    dm.id for dm in endpoint.list_models() if dm.display_name == "churn-v2"
][0]
v1_deployed_id = [
    dm.id for dm in endpoint.list_models() if dm.display_name == "churn-v1"
][0]

endpoint.update(traffic_split={v2_deployed_id: 100, v1_deployed_id: 0})
print("Promoted churn-v2 to 100% traffic.")
print("New traffic split:", endpoint.traffic_split)


IndexError: list index out of range


### 5.5 Rollback — Reverting to v1, No Retraining Required
**What this does:** simulates discovering a problem with v2 after promotion — the fix is an
**instant traffic-split revert**, not a new training run. This is the entire point of keeping v1
deployed (even at 0% traffic) rather than deleting it the moment v2 was promoted.


In [18]:

endpoint.update(traffic_split={v1_deployed_id: 100, v2_deployed_id: 0})
print("Rolled back to churn-v1 at 100% traffic.")
print("New traffic split:", endpoint.traffic_split)

for i in range(3):
    prediction = endpoint.predict(instances=[sample_instance])
    print(f"Post-rollback call {i+1}: deployed_model_id served = {prediction.deployed_model_id}")
    assert prediction.deployed_model_id == v1_deployed_id, "Expected v1 to be serving 100% of traffic"

print("\nConfirmed: 100% of traffic is back on churn-v1.")


NameError: name 'v1_deployed_id' is not defined


> 🖥️ **Check it in the console:** **Agent Platform > Models > Model Registry > churn-pipeline-model
> > Deploy & Test** → traffic split should now show 100% on `churn-v1`, 0% on `churn-v2` — the
> rollback, reflected instantly, with `churn-v2` still present (not deleted) in case you want to
> re-promote it later.



## 6. Model Monitoring

**What this does:** configures **Model Monitoring** on the endpoint to watch for **training-serving
skew** — when the statistical distribution of live prediction requests starts drifting away from
the distribution the model was actually trained on. Left undetected, this is one of the most common
ways a model silently degrades in production without anyone touching the model itself.


In [19]:

import pandas as pd

training_df = pd.read_csv(GITHUB_CSV_URL)
training_df.to_csv("training_data_for_monitoring.csv", index=False)
bucket.blob("monitoring/training_data_for_monitoring.csv").upload_from_filename(
    "training_data_for_monitoring.csv"
)
TRAINING_DATA_URI = f"{BUCKET_URI}/monitoring/training_data_for_monitoring.csv"
print("Uploaded training reference data to:", TRAINING_DATA_URI)


Uploaded training reference data to: gs://cs-poc-ytkjb6nscjlihtluxkcmuoq-mlops-lab-1783615448/monitoring/training_data_for_monitoring.csv


In [20]:

from google.cloud.aiplatform import model_monitoring

skew_config = model_monitoring.SkewDetectionConfig(
    data_source=TRAINING_DATA_URI,
    skew_thresholds={
        "income": 0.3,
        "tenure_months": 0.3,
        "num_purchases": 0.3,
    },
    target_field="churned",
)

objective_config = model_monitoring.ObjectiveConfig(skew_detection_config=skew_config)

monitoring_job = aiplatform.ModelDeploymentMonitoringJob.create(
    display_name="churn-endpoint-monitoring",
    endpoint=endpoint,
    logging_sampling_strategy=model_monitoring.RandomSampleConfig(sample_rate=1.0),
    schedule_config=model_monitoring.ScheduleConfig(monitor_interval=1),  # hours; smallest allowed
    objective_configs=objective_config,
    project=PROJECT_ID,
    location=REGION,
)
print("Monitoring job created:", monitoring_job.resource_name)


AttributeError: 'NoneType' object has no attribute 'as_proto'


### 6.1 Generate Some "Drifted" Prediction Traffic
**What this does:** sends prediction requests using values well outside the training data's
typical range — deliberately simulating drift (e.g. a much higher income bracket than anything in
the original 100-row dataset) so there's something for the monitoring job to actually flag.


In [21]:

import random

for _ in range(20):
    drifted_instance = [
        random.randint(60, 75),
        random.uniform(200000, 500000),  # far above the training data's income range
        random.choice(["East", "West"]),
        "Gold",
        random.randint(1, 12),
        random.randint(0, 3),
        random.uniform(1, 3),
    ]
    endpoint.predict(instances=[drifted_instance])

print("Sent 20 deliberately drifted prediction requests.")
print("Monitoring runs on an hourly schedule (the minimum interval) — skew results won't appear")
print("instantly; check back in the console after the next scheduled monitoring run.")


FailedPrecondition: 400 Endpoint projects/226858601047/locations/us-central1/endpoints/3303311711594872832 misconfigured, "traffic_split" not set.  Verify if any models are deployed to the endpoint and traffic split is configured for them.


> 🖥️ **Check it in the console:** **Agent Platform > Models > Model Monitoring** →
> `churn-endpoint-monitoring` → the **Feature skew** tab will show `income` (and possibly
> `tenure_months`/`num_purchases`) flagged once the next scheduled check runs, since the drifted
> requests above pushed those features well outside the training distribution's range.



## 7. Cleanup

This notebook created several continuously-billing resources: the monitoring job, the deployed
endpoint (with two model versions), the registered models, and the staging bucket. Guarded by
`CONFIRM_DELETE`, same pattern as the Day 4 notebook.


In [27]:

CONFIRM_DELETE = True  # <-- set to True to actually delete resources

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing will be deleted. "
          "Set CONFIRM_DELETE = True above and re-run this cell when you're ready.")


In [28]:

if CONFIRM_DELETE:
    try:
        monitoring_job.delete()
        print("Deleted monitoring job.")
    except Exception as e:
        print("Monitoring job cleanup skipped/failed:", e)


Monitoring job cleanup skipped/failed: name 'monitoring_job' is not defined


In [32]:

if CONFIRM_DELETE:
    try:
        endpoint.undeploy_all()
        endpoint.delete()
        print("Undeployed all models and deleted the endpoint.")
    except Exception as e:
        print("Endpoint cleanup skipped/failed:", e)


Endpoint cleanup skipped/failed: 400 There are other operations running on the Endpoint "projects/226858601047/locations/us-central1/endpoints/3303311711594872832". Operation(s) are: projects/226858601047/locations/us-central1/operations/4086139393293680640.


In [30]:

if CONFIRM_DELETE:
    matches = aiplatform.Model.list(filter='display_name="churn-pipeline-model"')
    for m in matches:
        try:
            m.delete()
            print(f"Deleted model version: {m.resource_name}")
        except Exception as e:
            print(f"Model cleanup skipped/failed for {m.resource_name}: {e}")


Model cleanup skipped/failed for projects/226858601047/locations/us-central1/models/6967790452925792256: 400 The model "projects/226858601047/locations/us-central1/models/6967790452925792256" can't be deleted because it's deployed or being deployed at the following endpoint(s): projects/226858601047/locations/us-central1/endpoints/3303311711594872832. Undeploy the model from all endpoints first and then delete it.


In [31]:

if CONFIRM_DELETE:
    for job in [pipeline_job_v1, pipeline_job_v2]:
        try:
            job.delete()
            print(f"Deleted pipeline job: {job.resource_name}")
        except Exception as e:
            print("Pipeline job cleanup skipped/failed:", e)

    try:
        bucket.delete(force=True)
        print(f"Deleted bucket {BUCKET_URI}.")
    except Exception as e:
        print("Bucket cleanup skipped/failed:", e)

    print("\nCleanup complete.")


Pipeline job cleanup skipped/failed: 400 The PipelineJob "projects/226858601047/locations/us-central1/pipelineJobs/churn-model-pipeline-20260709164904" is in state "RUNNING", and cannot be deleted. Please cancel it or wait for its completion before trying to delete it again.
Pipeline job cleanup skipped/failed: 400 The PipelineJob "projects/226858601047/locations/us-central1/pipelineJobs/churn-model-pipeline-20260709173159" is in state "RUNNING", and cannot be deleted. Please cancel it or wait for its completion before trying to delete it again.
Deleted bucket gs://cs-poc-ytkjb6nscjlihtluxkcmuoq-mlops-lab-1783615448.

Cleanup complete.



> 🖥️ **Final console check:** **Agent Platform > Models > Model Monitoring**, **Model Registry**,
> and **Deploy > Online prediction** should all show no resources remaining from this notebook.
